# 🗓️ Agentic RAG-Based Schedule Assistant

**Course project — Google Colab notebook**

This notebook builds an **agentic, Retrieval-Augmented Generation (RAG) schedule assistant** that:

- Stores a 30-day sample schedule (Aug 15, 2026 → Sep 13, 2026) with 6 event types.
- Keeps a **structured** source of truth in **SQLite** (for exact date/time operations).
- Keeps a **semantic** copy in **ChromaDB** (for natural-language retrieval), embedded with the **Google Gemini embedding model**.
- Uses the **Google Gemini API** as an **agent** with **native function/tool calling** to decide, on its own, whether to call:
  - `get_schedule` — retrieve schedule information (by date, time, date range, or free-text query)
  - `update_schedule` — add / update / delete events
- Demonstrates the full RAG pipeline, conflict detection, availability checking, and a **Gradio** chat UI.
- Ends with deployment instructions and a `deployed_url.txt` file.

> **Architecture in one line:** `User query → Gemini agent → (decides) → get_schedule (SQLite filter + ChromaDB semantic search) or update_schedule (SQLite write + ChromaDB sync) → Gemini generates final natural-language answer`

Run the cells **in order**, top to bottom.


In [ ]:
# CELL 2 — Install dependencies
# google-genai   -> the current official Google Gemini Python SDK
# chromadb       -> our vector database for semantic retrieval (RAG)
# pandas         -> for viewing the schedule as a table
# gradio         -> simple chat UI
# python-dateutil -> convenient date parsing helpers

!pip install -q google-genai chromadb pandas gradio python-dateutil
print("Dependencies installed.")


In [ ]:
# CELL 3 — Imports

import os
import json
import uuid
import sqlite3
from datetime import datetime, date, time, timedelta

import pandas as pd
import chromadb

from google import genai
from google.genai import types

import gradio as gr

print("Libraries imported successfully.")


In [ ]:
# CELL 4 — Set the Google Gemini API key
#
# SECURITY NOTE: never hard-code your real API key into a notebook you share.
#
# Preferred method (Google Colab Secrets):
#   1. Click the key icon (🔑) in the left sidebar of Colab ("Secrets").
#   2. Add a new secret named exactly:  GOOGLE_API_KEY
#   3. Paste your Gemini API key as the value and toggle "Notebook access" ON.
#   4. Re-run this cell.
#
# Fallback (only for local/non-shared environments):
#   os.environ["GOOGLE_API_KEY"] = "your-key-here"   # DO NOT commit this to a shared notebook
#
# This project intentionally does NOT use OPENAI_API_KEY anywhere.

GOOGLE_API_KEY = None

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY not found. Add it to Colab Secrets (recommended) or "
        "set os.environ['GOOGLE_API_KEY'] before running this cell."
    )

# Create a single Gemini client used everywhere in this notebook.
client = genai.Client(api_key=GOOGLE_API_KEY)

# We never print the key itself.
print("✅ Gemini client initialized successfully.")


In [ ]:
# CELL 5 — Configuration and current date
#
# Change these two model names here if your Google account / API key does not
# have access to the models below. Any Gemini model that supports function
# calling will work for GEMINI_MODEL, and any Gemini embedding model will
# work for EMBEDDING_MODEL (dimensions are handled automatically by ChromaDB
# since we always store the embeddings that this same model produces).

GEMINI_MODEL = "gemini-2.5-flash"        # supports native function/tool calling
EMBEDDING_MODEL = "gemini-embedding-001"  # Gemini embedding model for RAG
# Alternative embedding model if the above is unavailable: "text-embedding-004"

# We fix "today" for deterministic grading/testing, as required by the assignment.
CURRENT_DATE = date(2026, 8, 15)
CURRENT_DATE_STR = CURRENT_DATE.isoformat()

SQLITE_DB_PATH = "schedule.db"
CHROMA_DB_PATH = "./schedule_chroma_db"
CHROMA_COLLECTION_NAME = "schedule_events"

print(f"GEMINI_MODEL     = {GEMINI_MODEL}")
print(f"EMBEDDING_MODEL  = {EMBEDDING_MODEL}")
print(f"CURRENT_DATE     = {CURRENT_DATE_STR} (fixed for this assignment)")


In [ ]:
# CELL 6 — Generate the 30-day sample schedule
#
# We build 40+ realistic, non-identical events spread across
# Aug 15, 2026 -> Sep 13, 2026 (30 days), covering all 6 required event types.

import random
random.seed(42)  # reproducible sample data

EVENT_TYPES = ["Meeting", "Workshop", "Task", "Appointment", "Deadline", "Personal"]

EVENT_TEMPLATES = {
    "Meeting": [
        ("Team Meeting", "Weekly project sync with the engineering team."),
        ("Client Meeting", "Requirements discussion with the client."),
        ("Project Review", "Reviewing current sprint progress with stakeholders."),
        ("1:1 with Manager", "Monthly check-in and feedback session."),
        ("Hackathon Planning", "Planning logistics for the upcoming hackathon."),
    ],
    "Workshop": [
        ("Python Workshop", "Hands-on workshop covering advanced Python topics."),
        ("Machine Learning Workshop", "Introductory workshop on ML fundamentals."),
        ("Data Visualization Workshop", "Workshop on building dashboards and charts."),
        ("Cloud Computing Workshop", "Overview of cloud deployment best practices."),
    ],
    "Task": [
        ("Complete Project Report", "Finish writing the quarterly project report."),
        ("Study Session", "Focused study time for upcoming exams."),
        ("Code Review", "Review pending pull requests from the team."),
        ("Prepare Presentation Slides", "Build slides for the upcoming presentation."),
    ],
    "Appointment": [
        ("Doctor Appointment", "Routine check-up with the general physician."),
        ("Dentist Appointment", "Regular dental cleaning and check-up."),
        ("Personal Appointment", "Personal errand that requires a fixed time slot."),
        ("Interview", "Candidate interview for the open engineering role."),
    ],
    "Deadline": [
        ("Submit Assignment", "Final deadline to submit the course assignment."),
        ("Deadline: Report Submission", "Deadline for submitting the monthly report."),
        ("Deadline: Grant Proposal", "Final submission deadline for the grant proposal."),
    ],
    "Personal": [
        ("Gym Session", "Personal workout session."),
        ("Family Dinner", "Dinner plans with family."),
        ("Personal Appointment", "Personal time blocked on the calendar."),
        ("Weekend Trip Planning", "Planning logistics for a short weekend trip."),
    ],
}

START_TIMES = ["08:00", "09:00", "09:30", "10:00", "11:00", "12:30",
               "14:00", "15:00", "16:00", "17:00", "18:30"]
DURATIONS_MIN = [30, 45, 60, 90, 120]


def generate_sample_schedule(start_date: date, num_days: int = 30, num_events: int = 42):
    """Generate realistic, non-identical schedule events across `num_days` days."""
    events = []
    all_dates = [start_date + timedelta(days=i) for i in range(num_days)]

    for i in range(num_events):
        event_type = EVENT_TYPES[i % len(EVENT_TYPES)]
        title, description = random.choice(EVENT_TEMPLATES[event_type])
        event_date = random.choice(all_dates)

        start_str = random.choice(START_TIMES)
        start_dt = datetime.strptime(start_str, "%H:%M")
        duration = random.choice(DURATIONS_MIN)
        end_dt = start_dt + timedelta(minutes=duration)
        end_str = end_dt.strftime("%H:%M")

        events.append({
            "id": str(uuid.uuid4()),
            "title": title,
            "event_type": event_type,
            "date": event_date.isoformat(),
            "start_time": start_str,
            "end_time": end_str,
            "description": description,
        })

    # A few guaranteed, assignment-relevant events so demos are deterministic.
    guaranteed = [
        {"id": str(uuid.uuid4()), "title": "Team Meeting", "event_type": "Meeting",
         "date": (start_date + timedelta(days=1)).isoformat(), "start_time": "10:00",
         "end_time": "11:00", "description": "Weekly project discussion."},
        {"id": str(uuid.uuid4()), "title": "Project Meeting", "event_type": "Meeting",
         "date": (start_date + timedelta(days=4)).isoformat(), "start_time": "14:00",
         "end_time": "15:00", "description": "Discuss project milestones for next release."},
        {"id": str(uuid.uuid4()), "title": "Python Workshop", "event_type": "Workshop",
         "date": (start_date + timedelta(days=1)).isoformat(), "start_time": "13:00",
         "end_time": "16:00", "description": "Hands-on Python workshop for the team."},
    ]
    events.extend(guaranteed)

    events.sort(key=lambda e: (e["date"], e["start_time"]))
    return events


SCHEDULE_EVENTS = generate_sample_schedule(CURRENT_DATE, num_days=30, num_events=42)
print(f"Generated {len(SCHEDULE_EVENTS)} sample events across 30 days.")


In [ ]:
# CELL 7 — Display the generated schedule

schedule_df = pd.DataFrame(SCHEDULE_EVENTS)
schedule_df = schedule_df.sort_values(["date", "start_time"]).reset_index(drop=True)
print(f"Total events: {len(schedule_df)}")
print("Event type distribution:")
print(schedule_df["event_type"].value_counts())
schedule_df.head(20)


In [ ]:
# CELL 8 — Structured storage: SQLite (source of truth for exact operations)

def get_db_connection():
    conn = sqlite3.connect(SQLITE_DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS schedule (
            id TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            event_type TEXT NOT NULL,
            date TEXT NOT NULL,
            start_time TEXT NOT NULL,
            end_time TEXT NOT NULL,
            description TEXT
        )
    """)
    conn.commit()
    conn.close()


def db_insert_event(event: dict):
    conn = get_db_connection()
    conn.execute(
        "INSERT INTO schedule (id, title, event_type, date, start_time, end_time, description) "
        "VALUES (?, ?, ?, ?, ?, ?, ?)",
        (event["id"], event["title"], event["event_type"], event["date"],
         event["start_time"], event["end_time"], event.get("description", "")),
    )
    conn.commit()
    conn.close()


def db_update_event(event_id: str, updates: dict):
    conn = get_db_connection()
    fields, values = [], []
    for key in ["title", "event_type", "date", "start_time", "end_time", "description"]:
        if key in updates and updates[key] is not None:
            fields.append(f"{key} = ?")
            values.append(updates[key])
    if not fields:
        conn.close()
        return False
    values.append(event_id)
    conn.execute(f"UPDATE schedule SET {', '.join(fields)} WHERE id = ?", values)
    conn.commit()
    conn.close()
    return True


def db_delete_event(event_id: str):
    conn = get_db_connection()
    conn.execute("DELETE FROM schedule WHERE id = ?", (event_id,))
    conn.commit()
    conn.close()


def db_get_all_events():
    conn = get_db_connection()
    rows = conn.execute("SELECT * FROM schedule ORDER BY date, start_time").fetchall()
    conn.close()
    return [dict(r) for r in rows]


def db_get_event_by_id(event_id: str):
    conn = get_db_connection()
    row = conn.execute("SELECT * FROM schedule WHERE id = ?", (event_id,)).fetchone()
    conn.close()
    return dict(row) if row else None


# Initialize the DB and load the sample schedule (only if empty, so re-running is safe).
if os.path.exists(SQLITE_DB_PATH):
    os.remove(SQLITE_DB_PATH)  # fresh DB each run of this notebook for a clean demo
init_db()
for ev in SCHEDULE_EVENTS:
    db_insert_event(ev)

print(f"SQLite storage initialized with {len(db_get_all_events())} events at '{SQLITE_DB_PATH}'.")


In [ ]:
# CELL 9 — Initialize ChromaDB (vector database for semantic retrieval)

if os.path.exists(CHROMA_DB_PATH):
    import shutil
    shutil.rmtree(CHROMA_DB_PATH)  # fresh vector DB each run, matched to the fresh SQLite DB above

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_or_create_collection(name=CHROMA_COLLECTION_NAME)

print(f"ChromaDB collection '{CHROMA_COLLECTION_NAME}' ready at '{CHROMA_DB_PATH}'.")


In [ ]:
# CELL 10 — Create embeddings using the Gemini embedding model

def get_embedding(text: str, task_type: str = "RETRIEVAL_DOCUMENT"):
    """
    Turn a piece of text into a Gemini embedding vector.
    task_type = "RETRIEVAL_DOCUMENT" when embedding schedule events (things being stored),
    task_type = "RETRIEVAL_QUERY"    when embedding the user's question (thing doing the search).
    Using the correct task_type improves retrieval quality.
    """
    try:
        result = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        return result.embeddings[0].values
    except Exception as e:
        raise RuntimeError(
            f"Embedding request failed for model '{EMBEDDING_MODEL}'. "
            f"If your API key does not have access to this model, change "
            f"EMBEDDING_MODEL in CELL 5 (e.g. to 'text-embedding-004'). Original error: {e}"
        )


# Quick sanity check
sample_vector = get_embedding("Team Meeting on August 18, 2026")
print(f"Embedding created. Vector length: {len(sample_vector)}")


In [ ]:
# CELL 11 — Convert events to searchable text and insert into ChromaDB

def format_date_human(date_str: str) -> str:
    return datetime.strptime(date_str, "%Y-%m-%d").strftime("%B %d, %Y")


def format_time_human(time_str: str) -> str:
    return datetime.strptime(time_str, "%H:%M").strftime("%I:%M %p").lstrip("0")


def event_to_text(event: dict) -> str:
    """Convert a structured event into the searchable text format required by the assignment."""
    return (
        f"{event['title']} | {event['event_type']} | {format_date_human(event['date'])} | "
        f"{format_time_human(event['start_time'])} - {format_time_human(event['end_time'])} | "
        f"{event.get('description', '')}"
    )


def sync_event_to_chroma(event: dict):
    """Create/refresh the ChromaDB entry for a single event (used on add AND update)."""
    text = event_to_text(event)
    embedding = get_embedding(text, task_type="RETRIEVAL_DOCUMENT")
    collection.upsert(
        ids=[event["id"]],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{
            "title": event["title"],
            "event_type": event["event_type"],
            "date": event["date"],
            "start_time": event["start_time"],
            "end_time": event["end_time"],
        }],
    )


def remove_event_from_chroma(event_id: str):
    try:
        collection.delete(ids=[event_id])
    except Exception:
        pass  # already absent — safe to ignore


def rebuild_chroma_from_db():
    """Push every event currently in SQLite into ChromaDB with a fresh embedding."""
    events = db_get_all_events()
    for ev in events:
        sync_event_to_chroma(ev)
    return len(events)


num_synced = rebuild_chroma_from_db()
print(f"Synced {num_synced} events into ChromaDB with Gemini embeddings.")
print("Example searchable text:", event_to_text(SCHEDULE_EVENTS[0]))


In [ ]:
# CELL 12 — RAG step: semantic retrieval from ChromaDB
#
# This is the core of the RAG pipeline:
#   user query -> Gemini query embedding -> ChromaDB vector search -> relevant events

def retrieve_schedule_context(query: str, n_results: int = 5):
    """Embed `query` with Gemini and return the most semantically relevant schedule events."""
    query_embedding = get_embedding(query, task_type="RETRIEVAL_QUERY")
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)

    matches = []
    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    for i in range(len(ids)):
        matches.append({
            "id": ids[i],
            "document": docs[i],
            "metadata": metas[i],
            "distance": dists[i],
        })
    return matches


# Demonstrate the RAG pipeline explicitly
demo_matches = retrieve_schedule_context("upcoming workshop about Python", n_results=3)
print("RAG retrieval demo for query: 'upcoming workshop about Python'\n")
for m in demo_matches:
    print(f"- {m['document']}  (distance={m['distance']:.4f})")


In [ ]:
# CELL 13 — Date/time filtering utilities (structured, exact operations)
#
# Note: the Gemini agent (CELL 18) is given CURRENT_DATE in its system instructions and
# resolves relative expressions ("tomorrow", "next week", "Friday afternoon") into exact
# ISO dates / time ranges itself before calling get_schedule / update_schedule. The helper
# functions below back that up with reliable, deterministic Python logic and are also used
# directly by get_schedule, update_schedule and the conflict checker.

WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

TIME_PERIODS = {
    "morning": ("08:00", "12:00"),
    "afternoon": ("12:00", "17:00"),
    "evening": ("17:00", "21:00"),
}


def to_time(t_str: str) -> time:
    return datetime.strptime(t_str, "%H:%M").time()


def times_overlap(start1: str, end1: str, start2: str, end2: str) -> bool:
    """True if [start1,end1) overlaps [start2,end2)."""
    s1, e1, s2, e2 = to_time(start1), to_time(end1), to_time(start2), to_time(end2)
    return s1 < e2 and s2 < e1


def next_weekday_date(weekday_name: str, from_date: date = CURRENT_DATE) -> date:
    """Return the date of the next occurrence of a weekday (e.g. 'friday') on/after from_date."""
    weekday_name = weekday_name.lower()
    target_idx = WEEKDAYS.index(weekday_name)
    days_ahead = (target_idx - from_date.weekday()) % 7
    return from_date + timedelta(days=days_ahead)


def get_relative_date(keyword: str, from_date: date = CURRENT_DATE) -> date:
    keyword = keyword.lower().strip()
    if keyword == "today":
        return from_date
    if keyword == "tomorrow":
        return from_date + timedelta(days=1)
    if keyword == "yesterday":
        return from_date - timedelta(days=1)
    if keyword in WEEKDAYS:
        return next_weekday_date(keyword, from_date)
    raise ValueError(f"Unrecognized relative date keyword: {keyword}")


def get_next_week_range(from_date: date = CURRENT_DATE):
    """Return (start, end) ISO dates for 'next week' (the Mon-Sun following the current week)."""
    days_to_next_monday = (7 - from_date.weekday()) % 7 or 7
    next_monday = from_date + timedelta(days=days_to_next_monday)
    next_sunday = next_monday + timedelta(days=6)
    return next_monday.isoformat(), next_sunday.isoformat()


def get_weekend_range(from_date: date = CURRENT_DATE):
    """Return (start, end) ISO dates for 'this weekend' (upcoming/current Sat-Sun)."""
    days_to_sat = (5 - from_date.weekday()) % 7
    saturday = from_date + timedelta(days=days_to_sat)
    sunday = saturday + timedelta(days=1)
    return saturday.isoformat(), sunday.isoformat()


def events_on_date(events, date_str):
    return [e for e in events if e["date"] == date_str]


def events_in_range(events, start_date, end_date):
    return [e for e in events if start_date <= e["date"] <= end_date]


def events_in_time_range(events, start_time, end_time):
    return [e for e in events if times_overlap(e["start_time"], e["end_time"], start_time, end_time)]


print("Date/time utilities ready. Example: Friday's date ->", next_weekday_date("friday"))
print("Example: 'next week' range ->", get_next_week_range())


In [ ]:
# CELL 14 — TOOL 1: get_schedule

def get_schedule(date: str = None, date_range_start: str = None, date_range_end: str = None,
                  start_time: str = None, end_time: str = None, query: str = None):
    """
    Retrieve schedule information.

    Args (all optional — combine as many as are known):
        date: exact ISO date 'YYYY-MM-DD' to look up.
        date_range_start / date_range_end: ISO dates defining an inclusive date range.
        start_time / end_time: 'HH:MM' 24-hour time window to filter within a date/date range.
        query: free-text natural-language description used for semantic (RAG) search,
               e.g. 'python workshop', 'next project meeting'.

    Returns:
        dict with 'count' and 'events' (each event has id, title, event_type, date,
        start_time, end_time, description).
    """
    all_events = db_get_all_events()
    results = all_events
    structured_filter_used = False

    if date:
        results = events_on_date(results, date)
        structured_filter_used = True
    elif date_range_start and date_range_end:
        results = events_in_range(results, date_range_start, date_range_end)
        structured_filter_used = True

    if start_time and end_time:
        results = events_in_time_range(results, start_time, end_time)
        structured_filter_used = True

    if query:
        if structured_filter_used:
            # Refine the already-filtered structured results using semantic similarity,
            # so exact date/time constraints are always respected.
            semantic_matches = retrieve_schedule_context(query, n_results=10)
            semantic_ids = {m["id"] for m in semantic_matches}
            refined = [e for e in results if e["id"] in semantic_ids]
            results = refined if refined else results
        else:
            # Pure natural-language query -> rely fully on RAG retrieval.
            semantic_matches = retrieve_schedule_context(query, n_results=5)
            semantic_ids = [m["id"] for m in semantic_matches]
            id_to_event = {e["id"]: e for e in all_events}
            results = [id_to_event[i] for i in semantic_ids if i in id_to_event]

    results = sorted(results, key=lambda e: (e["date"], e["start_time"]))
    return {"count": len(results), "events": results}


# Sanity check
print(get_schedule(date=CURRENT_DATE_STR))


In [ ]:
# CELL 15 — TOOL 2: update_schedule

def update_schedule(action: str, event_id: str = None, title: str = None, event_type: str = None,
                     date: str = None, start_time: str = None, end_time: str = None,
                     description: str = None, force: bool = False):
    """
    Add, update, or delete a schedule event. Keeps SQLite and ChromaDB in sync.

    Args:
        action: one of 'add', 'update', 'delete'.
        event_id: required for 'update' and 'delete' — the exact event to modify/remove.
                  If unknown, call get_schedule first to look up the correct id.
        title, event_type, date, start_time, end_time, description:
            fields to set (for 'add', title/event_type/date/start_time/end_time are required;
            for 'update', only the fields being changed need to be provided).
        force: if True, skip conflict checking and create/move the event anyway.

    Returns:
        dict describing the outcome, including a 'conflicts' list when a conflict blocks
        the operation.
    """
    action = action.lower().strip()

    if action == "add":
        missing = [f for f in ["title", "event_type", "date", "start_time", "end_time"]
                   if not locals()[f]]
        if missing:
            return {"status": "error", "message": f"Missing required fields to add event: {missing}"}

        if not force:
            conflicts = check_conflict(date, start_time, end_time)
            if conflicts:
                return {
                    "status": "conflict",
                    "message": "The requested time overlaps with an existing event.",
                    "conflicts": conflicts,
                }

        new_event = {
            "id": str(uuid.uuid4()),
            "title": title,
            "event_type": event_type,
            "date": date,
            "start_time": start_time,
            "end_time": end_time,
            "description": description or "",
        }
        db_insert_event(new_event)
        sync_event_to_chroma(new_event)
        return {"status": "success", "message": "Event added.", "event": new_event}

    elif action == "update":
        if not event_id:
            return {"status": "error", "message": "event_id is required to update an event. "
                                                    "Use get_schedule first to find it."}
        existing = db_get_event_by_id(event_id)
        if not existing:
            return {"status": "error", "message": f"No event found with id {event_id}."}

        updates = {k: v for k, v in {
            "title": title, "event_type": event_type, "date": date,
            "start_time": start_time, "end_time": end_time, "description": description,
        }.items() if v is not None}

        merged = {**existing, **updates}

        if not force and (date or start_time or end_time):
            conflicts = check_conflict(merged["date"], merged["start_time"], merged["end_time"],
                                        exclude_id=event_id)
            if conflicts:
                return {
                    "status": "conflict",
                    "message": "The new time overlaps with an existing event.",
                    "conflicts": conflicts,
                }

        db_update_event(event_id, updates)
        updated_event = db_get_event_by_id(event_id)
        sync_event_to_chroma(updated_event)  # refresh vector entry so retrieval isn't stale
        return {"status": "success", "message": "Event updated.", "event": updated_event}

    elif action == "delete":
        if not event_id:
            return {"status": "error", "message": "event_id is required to delete an event. "
                                                    "Use get_schedule first to find it."}
        existing = db_get_event_by_id(event_id)
        if not existing:
            return {"status": "error", "message": f"No event found with id {event_id}."}

        db_delete_event(event_id)
        remove_event_from_chroma(event_id)  # keep ChromaDB free of stale entries
        return {"status": "success", "message": "Event deleted.", "event": existing}

    else:
        return {"status": "error", "message": f"Unknown action '{action}'. Use add/update/delete."}


print("update_schedule tool ready.")


In [ ]:
# CELL 16 — Conflict detection

def check_conflict(date_str: str, start_time: str, end_time: str, exclude_id: str = None):
    """Return a list of existing events on `date_str` that overlap [start_time, end_time)."""
    same_day_events = events_on_date(db_get_all_events(), date_str)
    conflicts = []
    for e in same_day_events:
        if exclude_id and e["id"] == exclude_id:
            continue
        if times_overlap(e["start_time"], e["end_time"], start_time, end_time):
            conflicts.append(e)
    return conflicts


def check_availability(date_str: str, start_time: str, end_time: str):
    """Return a free/busy breakdown of [start_time, end_time) on `date_str`."""
    conflicts = sorted(check_conflict(date_str, start_time, end_time), key=lambda e: e["start_time"])
    if not conflicts:
        return {
            "is_free": True,
            "busy_events": [],
            "message": f"You are completely free on {date_str} from "
                       f"{format_time_human(start_time)} to {format_time_human(end_time)}.",
        }

    free_slots = []
    cursor = to_time(start_time)
    window_end = to_time(end_time)
    for e in conflicts:
        busy_start, busy_end = to_time(e["start_time"]), to_time(e["end_time"])
        if cursor < busy_start:
            free_slots.append((cursor, min(busy_start, window_end)))
        cursor = max(cursor, busy_end)
    if cursor < window_end:
        free_slots.append((cursor, window_end))

    free_desc = ", ".join(
        f"{format_time_human(s.strftime('%H:%M'))} to {format_time_human(e.strftime('%H:%M'))}"
        for s, e in free_slots if s < e
    ) or "no free time"
    busy_desc = "; ".join(
        f"{e['title']} from {format_time_human(e['start_time'])} to {format_time_human(e['end_time'])}"
        for e in conflicts
    )
    return {
        "is_free": False,
        "busy_events": conflicts,
        "message": f"You are free from {free_desc}. You have: {busy_desc}.",
    }


print("Conflict detection ready. Example:", check_conflict(CURRENT_DATE_STR, "09:00", "10:00"))


In [ ]:
# CELL 17 — Gemini function/tool schema definitions for get_schedule and update_schedule
#
# These schemas are given to Gemini so IT decides (via native function calling) which
# tool to call and with what arguments — no keyword matching is used anywhere.

get_schedule_declaration = types.FunctionDeclaration(
    name="get_schedule",
    description=(
        "Retrieve schedule events. Use this whenever the user asks what is on their "
        "schedule, whether they are free/busy, or wants to find a specific event. "
        "Resolve relative dates ('tomorrow', 'next Friday', 'next week') into exact "
        "'YYYY-MM-DD' values yourself before calling, using the current date provided "
        "in your instructions."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "date": types.Schema(type=types.Type.STRING,
                                  description="Exact date to look up, format YYYY-MM-DD."),
            "date_range_start": types.Schema(type=types.Type.STRING,
                                              description="Start of a date range, format YYYY-MM-DD."),
            "date_range_end": types.Schema(type=types.Type.STRING,
                                            description="End of a date range, format YYYY-MM-DD."),
            "start_time": types.Schema(type=types.Type.STRING,
                                        description="Start of a time window, 24-hour HH:MM."),
            "end_time": types.Schema(type=types.Type.STRING,
                                      description="End of a time window, 24-hour HH:MM."),
            "query": types.Schema(type=types.Type.STRING,
                                   description="Free-text description of what the user is looking for."),
        },
    ),
)

update_schedule_declaration = types.FunctionDeclaration(
    name="update_schedule",
    description=(
        "Add, update, or delete a schedule event. For 'update' or 'delete', you must "
        "know the event's id — call get_schedule first to find the correct event, then "
        "call update_schedule with that event_id."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "action": types.Schema(type=types.Type.STRING,
                                    description="One of: add, update, delete."),
            "event_id": types.Schema(type=types.Type.STRING,
                                      description="The id of the event to update/delete."),
            "title": types.Schema(type=types.Type.STRING, description="Event title."),
            "event_type": types.Schema(type=types.Type.STRING,
                                        description="One of: Meeting, Workshop, Task, Appointment, Deadline, Personal."),
            "date": types.Schema(type=types.Type.STRING, description="Event date, format YYYY-MM-DD."),
            "start_time": types.Schema(type=types.Type.STRING, description="Start time, 24-hour HH:MM."),
            "end_time": types.Schema(type=types.Type.STRING, description="End time, 24-hour HH:MM."),
            "description": types.Schema(type=types.Type.STRING, description="Event description."),
        },
        required=["action"],
    ),
)

schedule_tool = types.Tool(function_declarations=[get_schedule_declaration, update_schedule_declaration])

print("Gemini tool schemas defined: get_schedule, update_schedule")


In [ ]:
# CELL 18 — The Gemini agent / tool-calling loop
#
# Full flow: user message -> Gemini (with tools) -> Gemini requests a function call
# -> we execute the real Python function -> we send the result back to Gemini
# -> Gemini writes the final natural-language answer.
#
# NOTE (added after real-world testing): the free tier of the Gemini API can return
# transient 503 (model overloaded) or 429 (rate limit exceeded) errors, especially on
# models Google has recently auto-migrated free accounts onto. generate_with_retry()
# below retries those specific errors with exponential backoff before giving up.

SYSTEM_INSTRUCTION = f"""You are a helpful schedule assistant.

Today's date is {CURRENT_DATE_STR} ({CURRENT_DATE.strftime('%A, %B %d, %Y')}). Always resolve
relative dates and times (today, tomorrow, yesterday, weekday names, 'next week',
'this weekend', 'morning'/'afternoon'/'evening') into exact values relative to this date
before calling a tool. Afternoon = 12:00-17:00, morning = 08:00-12:00, evening = 17:00-21:00.
All dates you pass to tools must be 'YYYY-MM-DD' and all times '24-hour HH:MM'.

You have two tools:
- get_schedule: for any question about what is scheduled, availability/free-time
  questions, or finding a specific event.
- update_schedule: for adding, moving/changing, or cancelling/deleting events. If the
  request is a delete/update and you don't already know the event_id, call get_schedule
  first to find the right event, then call update_schedule.

If update_schedule reports a conflict, explain the conflict clearly to the user and ask
whether they'd like a different time, rather than assuming.

After a tool returns a result, answer the user in clear, natural language. For
availability questions, explicitly state the free time ranges as well as any busy
periods (do not just say yes/no).
"""


def generate_with_retry(contents, config, max_retries: int = 5):
    """
    Call Gemini, automatically retrying on transient 503 (overloaded) or 429 (rate
    limit) errors with exponential backoff. Raises the original error if it is not
    one of those, or if retries are exhausted.
    """
    delay = 5  # seconds, doubles each retry
    last_error = None
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            is_retryable = "503" in msg or "UNAVAILABLE" in msg or "429" in msg or "RESOURCE_EXHAUSTED" in msg
            if not is_retryable or attempt == max_retries - 1:
                raise
            last_error = e
            print(f"  ...Gemini returned a transient error (attempt {attempt + 1}/{max_retries}), "
                  f"retrying in {delay}s: {msg[:120]}")
            time_module.sleep(delay)
            delay *= 2
    raise last_error


def call_tool(name: str, args: dict):
    """Dispatch a Gemini function-call request to the matching real Python function."""
    if name == "get_schedule":
        return get_schedule(**args)
    elif name == "update_schedule":
        return update_schedule(**args)
    else:
        return {"status": "error", "message": f"Unknown tool '{name}'."}


def run_agent(user_query: str, verbose: bool = False):
    """Run one full agentic turn: Gemini decides whether/which tool to call, we execute
    it, and Gemini produces the final answer. Returns a dict with full trace info."""
    try:
        config = types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=[schedule_tool],
        )
        contents = [types.Content(role="user", parts=[types.Part(text=user_query)])]

        response = generate_with_retry(contents, config)

        function_call_part = None
        for part in response.candidates[0].content.parts:
            if getattr(part, "function_call", None):
                function_call_part = part
                break

        if function_call_part is None:
            return {
                "user_query": user_query, "tool_called": None, "tool_args": None,
                "tool_result": None, "final_response": response.text,
            }

        fn_name = function_call_part.function_call.name
        fn_args = dict(function_call_part.function_call.args)
        if verbose:
            print(f"[agent] Gemini chose tool: {fn_name}({fn_args})")

        tool_result = call_tool(fn_name, fn_args)

        # Send the model's function call + our function's result back to Gemini
        # so it can compose the final natural-language answer.
        contents.append(response.candidates[0].content)
        function_response_part = types.Part.from_function_response(
            name=fn_name, response={"result": tool_result},
        )
        contents.append(types.Content(role="user", parts=[function_response_part]))

        final_response = generate_with_retry(contents, config)

        return {
            "user_query": user_query, "tool_called": fn_name, "tool_args": fn_args,
            "tool_result": tool_result, "final_response": final_response.text,
        }

    except Exception as e:
        return {
            "user_query": user_query, "tool_called": None, "tool_args": None,
            "tool_result": None,
            "final_response": f"⚠️ The assistant hit an error talking to Gemini: {e}",
        }


print("Agent loop ready (with automatic retry on transient 503/429 errors).")


In [ ]:
# CELL 19 — Test: simple retrieval (RAG pipeline directly)

result = get_schedule(query="python workshop")
print(json.dumps(result, indent=2))


In [ ]:
# CELL 20 — Test: adding an event (structured tool directly)

add_result = update_schedule(
    action="add", title="Design Sync", event_type="Meeting",
    date=(CURRENT_DATE + timedelta(days=2)).isoformat(),
    start_time="09:00", end_time="09:30", description="Quick sync on UI design.",
)
print(json.dumps(add_result, indent=2))


In [ ]:
# CELL 21 — Test: updating an event (structured tool directly)

target = get_schedule(query="Design Sync")["events"][0]
update_result = update_schedule(
    action="update", event_id=target["id"], start_time="11:00", end_time="11:30",
)
print(json.dumps(update_result, indent=2))


In [ ]:
# CELL 22 — Test: deleting an event (structured tool directly)

delete_result = update_schedule(action="delete", event_id=target["id"])
print(json.dumps(delete_result, indent=2))

# Confirm it's really gone from both stores
print("Still in SQLite?", db_get_event_by_id(target["id"]))
print("Still in ChromaDB?", collection.get(ids=[target["id"]])["ids"])


In [ ]:
# CELL 23 — Test: availability checking

friday = next_weekday_date("friday")
afternoon_start, afternoon_end = TIME_PERIODS["afternoon"]
availability = check_availability(friday.isoformat(), afternoon_start, afternoon_end)
print(availability["message"])


In [ ]:
# CELL 24 — Run ALL required assignment test cases through the Gemini agent
#
# For each test we print: user query, tool selected, tool arguments, tool result, and
# the final Gemini response — exactly as required for the demonstration.
#
# NOTE (added after real-world testing): each test can make up to 2 Gemini calls, and
# the free tier for some models allows as few as 5 requests/minute. We add a short pause
# between tests so this loop doesn't blow through that quota. If you still see 429
# errors, increase SECONDS_BETWEEN_TESTS below, or check your current limits/model at
# https://ai.google.dev/gemini-api/docs/rate-limits

SECONDS_BETWEEN_TESTS = 15

REQUIRED_TEST_QUERIES = [
    "What do I have scheduled tomorrow?",
    "Am I free Friday afternoon?",
    "Add a meeting on August 15 at 3 PM.",
    "Move my meeting from 2 PM to 4 PM.",
    "What appointments do I have next week?",
    "Do I have anything scheduled on August 20?",
    "Cancel my workshop tomorrow.",
    "When is my next project meeting?",
    "Add a Python workshop on August 25 from 2 PM to 4 PM.",
    "Am I free between 10 AM and 1 PM on August 22?",
]

for i, q in enumerate(REQUIRED_TEST_QUERIES, start=1):
    print(f"{'='*90}\nTest {i}: {q}\n{'='*90}")
    result = run_agent(q)
    print(f"Tool selected : {result['tool_called']}")
    print(f"Tool arguments: {result['tool_args']}")
    tool_result_preview = json.dumps(result['tool_result'])[:400] if result['tool_result'] else None
    print(f"Tool result   : {tool_result_preview}")
    print(f"Final answer  : {result['final_response']}\n")

    if i < len(REQUIRED_TEST_QUERIES):
        time_module.sleep(SECONDS_BETWEEN_TESTS)


In [ ]:
# CELL 25 — Interactive chat UI (Gradio)

def chat_fn(message, history):
    result = run_agent(message)
    return result["final_response"]


with gr.Blocks(title="Agentic RAG Schedule Assistant") as demo:
    gr.Markdown("# 🗓️ Agentic RAG Schedule Assistant")
    gr.Markdown(
        "Ask about your schedule (e.g. *'What do I have tomorrow?'*, *'Am I free Friday afternoon?'*) "
        "or ask me to change it (e.g. *'Add a meeting on August 20 at 3 PM'*, *'Cancel my workshop tomorrow'*). "
        "Powered by the Google Gemini API with agentic tool calling over a RAG pipeline (ChromaDB + SQLite)."
    )
    gr.ChatInterface(fn=chat_fn, type="messages")

# share=True gives a temporary public URL (valid ~72 hours) — useful for live demos,
# but NOT a permanent deployment. See CELL 28/29 for real deployment.
# demo.launch(share=True)
demo.launch()


In [ ]:
# CELL 26 — Demonstrate several user interactions (without the UI, for the report/output log)

DEMO_QUERIES = [
    "What's on my schedule tomorrow?",
    "Add a meeting tomorrow at 3 PM called Budget Review.",
    "Do I have any deadlines this week?",
    "Am I free on August 22 in the morning?",
]

for q in DEMO_QUERIES:
    result = run_agent(q, verbose=True)
    print(f"User: {q}")
    print(f"Assistant: {result['final_response']}\n")


## 🏗️ Architecture and RAG Flow — Explanation

**Two storage layers, working together:**

1. **SQLite (`schedule.db`)** — the *structured source of truth*. Every exact operation
   (does this time slot already have an event? move this specific meeting; delete this
   specific appointment) is done here, because vector similarity search is not reliable
   for exact date/time logic.
2. **ChromaDB (`schedule_chroma_db/`)** — the *semantic index* used for RAG. Every event is
   converted to a sentence like `"Team Meeting | Meeting | August 18, 2026 | 10:00 AM - 11:00 AM | Weekly project discussion"`,
   embedded with the Gemini embedding model, and stored as a vector. This lets the
   assistant answer fuzzy questions like *"when is my next project meeting"* where the
   user doesn't give an exact date.

**The RAG pipeline (CELL 12 / CELL 14):**

```
user query → Gemini embedding (RETRIEVAL_QUERY) → ChromaDB similarity search
           → top-k relevant events → combined with structured filters (date/time)
           → returned to the agent as tool output → Gemini writes the final answer
```

**Why this is *agentic*, not just RAG:**

The Gemini model is given **native function/tool declarations** for `get_schedule` and
`update_schedule` (CELL 17) and a system instruction describing when to use each
(CELL 18). Gemini — not our Python code — decides which tool to call and with what
arguments, based on understanding the user's natural-language request. Our code never
inspects the text for keywords like "add" or "delete"; it only executes whichever
function Gemini asked for and reports the result back so Gemini can compose the final
reply. This closes the full agent loop: **reason → act (tool call) → observe (tool
result) → respond**.

**Data consistency:** every `update_schedule` call writes to SQLite *and* immediately
calls `sync_event_to_chroma` (for add/update) or `remove_event_from_chroma` (for delete),
so ChromaDB never serves stale results after a modification.


In [ ]:
# CELL 28 — Prepare the application for deployment (Hugging Face Spaces)
#
# Google Colab itself is NOT a permanent deployment: `demo.launch()` only runs while this
# notebook's runtime is alive, and `share=True` only gives a temporary link (~72 hours).
# Below we generate a standalone `app.py` + `requirements.txt` that can be deployed to a
# free, persistent host. Hugging Face Spaces (Gradio SDK) is used here because it is the
# simplest beginner-friendly option that still uses the Gemini API (not OpenAI) and keeps
# the API key secure via a Space "secret".

os.makedirs("deployment", exist_ok=True)

requirements_txt = """google-genai
chromadb
pandas
gradio
python-dateutil
"""
with open("deployment/requirements.txt", "w") as f:
    f.write(requirements_txt)

# app.py bundles the same logic used above (config, sample data, SQLite, ChromaDB,
# RAG retrieval, get_schedule, update_schedule, conflict detection, the Gemini agent
# loop, and the Gradio UI) into one self-contained script suitable for Spaces.
app_py = '''import os
import json
import uuid
import sqlite3
import random
from datetime import datetime, date, time, timedelta

import chromadb
from google import genai
from google.genai import types
import gradio as gr

GEMINI_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"
CURRENT_DATE = date(2026, 8, 15)
CURRENT_DATE_STR = CURRENT_DATE.isoformat()
SQLITE_DB_PATH = "schedule.db"
CHROMA_DB_PATH = "./schedule_chroma_db"
CHROMA_COLLECTION_NAME = "schedule_events"

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Set the GOOGLE_API_KEY secret in your Space settings.")
client = genai.Client(api_key=GOOGLE_API_KEY)

EVENT_TYPES = ["Meeting", "Workshop", "Task", "Appointment", "Deadline", "Personal"]
random.seed(42)

def get_db_connection():
    conn = sqlite3.connect(SQLITE_DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db_connection()
    conn.execute("CREATE TABLE IF NOT EXISTS schedule ("
                 "id TEXT PRIMARY KEY, title TEXT NOT NULL, event_type TEXT NOT NULL, "
                 "date TEXT NOT NULL, start_time TEXT NOT NULL, end_time TEXT NOT NULL, description TEXT)")
    conn.commit(); conn.close()

def db_insert_event(e):
    conn = get_db_connection()
    conn.execute("INSERT OR REPLACE INTO schedule VALUES (?,?,?,?,?,?,?)",
                 (e["id"], e["title"], e["event_type"], e["date"], e["start_time"], e["end_time"], e.get("description","")))
    conn.commit(); conn.close()

def db_update_event(event_id, updates):
    conn = get_db_connection()
    fields, values = [], []
    for k in ["title","event_type","date","start_time","end_time","description"]:
        if updates.get(k) is not None:
            fields.append(f"{k} = ?"); values.append(updates[k])
    if fields:
        values.append(event_id)
        conn.execute(f"UPDATE schedule SET {', '.join(fields)} WHERE id = ?", values)
        conn.commit()
    conn.close()

def db_delete_event(event_id):
    conn = get_db_connection(); conn.execute("DELETE FROM schedule WHERE id = ?", (event_id,)); conn.commit(); conn.close()

def db_get_all_events():
    conn = get_db_connection()
    rows = conn.execute("SELECT * FROM schedule ORDER BY date, start_time").fetchall()
    conn.close(); return [dict(r) for r in rows]

def db_get_event_by_id(event_id):
    conn = get_db_connection()
    row = conn.execute("SELECT * FROM schedule WHERE id = ?", (event_id,)).fetchone()
    conn.close(); return dict(row) if row else None

def seed_sample_events():
    if db_get_all_events():
        return
    templates = {
        "Meeting": ("Team Meeting", "Weekly project discussion."),
        "Workshop": ("Python Workshop", "Hands-on Python workshop for the team."),
        "Task": ("Complete Project Report", "Finish writing the quarterly report."),
        "Appointment": ("Doctor Appointment", "Routine check-up."),
        "Deadline": ("Submit Assignment", "Final deadline to submit the assignment."),
        "Personal": ("Gym Session", "Personal workout session."),
    }
    for i in range(30):
        d = CURRENT_DATE + timedelta(days=i % 30)
        etype = EVENT_TYPES[i % len(EVENT_TYPES)]
        title, desc = templates[etype]
        start_h = 8 + (i % 10)
        db_insert_event({"id": str(uuid.uuid4()), "title": title, "event_type": etype,
                          "date": d.isoformat(), "start_time": f"{start_h:02d}:00",
                          "end_time": f"{start_h+1:02d}:00", "description": desc})

init_db(); seed_sample_events()

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_or_create_collection(name=CHROMA_COLLECTION_NAME)

def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    r = client.models.embed_content(model=EMBEDDING_MODEL, contents=text,
                                     config=types.EmbedContentConfig(task_type=task_type))
    return r.embeddings[0].values

def event_to_text(e):
    d = datetime.strptime(e["date"], "%Y-%m-%d").strftime("%B %d, %Y")
    st = datetime.strptime(e["start_time"], "%H:%M").strftime("%I:%M %p").lstrip("0")
    et = datetime.strptime(e["end_time"], "%H:%M").strftime("%I:%M %p").lstrip("0")
    return f"{e['title']} | {e['event_type']} | {d} | {st} - {et} | {e.get('description', '')}"

def sync_event_to_chroma(e):
    text = event_to_text(e)
    emb = get_embedding(text, "RETRIEVAL_DOCUMENT")
    collection.upsert(ids=[e["id"]], embeddings=[emb], documents=[text],
                       metadatas={"date": e["date"], "event_type": e["event_type"]})

def remove_event_from_chroma(event_id):
    try: collection.delete(ids=[event_id])
    except Exception: pass

for e in db_get_all_events():
    sync_event_to_chroma(e)

def retrieve_schedule_context(query, n_results=5):
    emb = get_embedding(query, "RETRIEVAL_QUERY")
    res = collection.query(query_embeddings=[emb], n_results=n_results)
    return res.get("ids", [[]])[0]

def to_time(t): return datetime.strptime(t, "%H:%M").time()
def times_overlap(s1,e1,s2,e2):
    a,b,c,d = to_time(s1), to_time(e1), to_time(s2), to_time(e2)
    return a < d and c < b

def check_conflict(date_str, start_time, end_time, exclude_id=None):
    same_day = [e for e in db_get_all_events() if e["date"] == date_str]
    return [e for e in same_day if e["id"] != exclude_id and times_overlap(e["start_time"], e["end_time"], start_time, end_time)]

def get_schedule(date=None, date_range_start=None, date_range_end=None, start_time=None, end_time=None, query=None):
    events = db_get_all_events()
    results = events
    used = False
    if date:
        results = [e for e in results if e["date"] == date]; used = True
    elif date_range_start and date_range_end:
        results = [e for e in results if date_range_start <= e["date"] <= date_range_end]; used = True
    if start_time and end_time:
        results = [e for e in results if times_overlap(e["start_time"], e["end_time"], start_time, end_time)]; used = True
    if query:
        ids = set(retrieve_schedule_context(query, 10))
        if used:
            refined = [e for e in results if e["id"] in ids]
            results = refined if refined else results
        else:
            id_map = {e["id"]: e for e in events}
            results = [id_map[i] for i in ids if i in id_map]
    results = sorted(results, key=lambda e: (e["date"], e["start_time"]))
    return {"count": len(results), "events": results}

def update_schedule(action, event_id=None, title=None, event_type=None, date=None,
                     start_time=None, end_time=None, description=None, force=False):
    action = action.lower().strip()
    if action == "add":
        if not all([title, event_type, date, start_time, end_time]):
            return {"status": "error", "message": "Missing required fields."}
        if not force:
            c = check_conflict(date, start_time, end_time)
            if c: return {"status": "conflict", "conflicts": c}
        ev = {"id": str(uuid.uuid4()), "title": title, "event_type": event_type, "date": date,
              "start_time": start_time, "end_time": end_time, "description": description or ""}
        db_insert_event(ev); sync_event_to_chroma(ev)
        return {"status": "success", "event": ev}
    elif action == "update":
        if not event_id: return {"status": "error", "message": "event_id required."}
        existing = db_get_event_by_id(event_id)
        if not existing: return {"status": "error", "message": "Event not found."}
        updates = {k: v for k, v in {"title": title, "event_type": event_type, "date": date,
                   "start_time": start_time, "end_time": end_time, "description": description}.items() if v is not None}
        merged = {**existing, **updates}
        if not force and (date or start_time or end_time):
            c = check_conflict(merged["date"], merged["start_time"], merged["end_time"], event_id)
            if c: return {"status": "conflict", "conflicts": c}
        db_update_event(event_id, updates)
        updated = db_get_event_by_id(event_id)
        sync_event_to_chroma(updated)
        return {"status": "success", "event": updated}
    elif action == "delete":
        if not event_id: return {"status": "error", "message": "event_id required."}
        existing = db_get_event_by_id(event_id)
        if not existing: return {"status": "error", "message": "Event not found."}
        db_delete_event(event_id); remove_event_from_chroma(event_id)
        return {"status": "success", "event": existing}
    return {"status": "error", "message": "Unknown action."}

get_schedule_declaration = types.FunctionDeclaration(
    name="get_schedule", description="Retrieve schedule events by date, date range, time, and/or free-text query.",
    parameters=types.Schema(type=types.Type.OBJECT, properties={
        "date": types.Schema(type=types.Type.STRING), "date_range_start": types.Schema(type=types.Type.STRING),
        "date_range_end": types.Schema(type=types.Type.STRING), "start_time": types.Schema(type=types.Type.STRING),
        "end_time": types.Schema(type=types.Type.STRING), "query": types.Schema(type=types.Type.STRING)}))

update_schedule_declaration = types.FunctionDeclaration(
    name="update_schedule", description="Add, update, or delete a schedule event.",
    parameters=types.Schema(type=types.Type.OBJECT, properties={
        "action": types.Schema(type=types.Type.STRING), "event_id": types.Schema(type=types.Type.STRING),
        "title": types.Schema(type=types.Type.STRING), "event_type": types.Schema(type=types.Type.STRING),
        "date": types.Schema(type=types.Type.STRING), "start_time": types.Schema(type=types.Type.STRING),
        "end_time": types.Schema(type=types.Type.STRING), "description": types.Schema(type=types.Type.STRING)},
        required=["action"]))

schedule_tool = types.Tool(function_declarations=[get_schedule_declaration, update_schedule_declaration])

SYSTEM_INSTRUCTION = f"You are a helpful schedule assistant. Today is {CURRENT_DATE_STR}. Resolve relative dates/times to exact YYYY-MM-DD / HH:MM before calling tools. Afternoon=12:00-17:00, morning=08:00-12:00, evening=17:00-21:00. Use get_schedule for lookups/availability, update_schedule for add/update/delete (look up event_id via get_schedule first if needed). Explain conflicts clearly."

def call_tool(name, args):
    if name == "get_schedule": return get_schedule(**args)
    if name == "update_schedule": return update_schedule(**args)
    return {"status": "error", "message": "unknown tool"}

def run_agent(user_query):
    config = types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTION, tools=[schedule_tool])
    contents = [types.Content(role="user", parts=[types.Part(text=user_query)])]
    response = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
    fc_part = None
    for part in response.candidates[0].content.parts:
        if getattr(part, "function_call", None):
            fc_part = part; break
    if fc_part is None:
        return response.text
    fn_name = fc_part.function_call.name
    fn_args = dict(fc_part.function_call.args)
    tool_result = call_tool(fn_name, fn_args)
    contents.append(response.candidates[0].content)
    contents.append(types.Content(role="user", parts=[types.Part.from_function_response(name=fn_name, response={"result": tool_result})]))
    final = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
    return final.text

def chat_fn(message, history):
    try:
        return run_agent(message)
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks(title="Agentic RAG Schedule Assistant") as demo:
    gr.Markdown("# Agentic RAG Schedule Assistant")
    gr.ChatInterface(fn=chat_fn, type="messages")

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", 7860)))
'''

with open("deployment/app.py", "w") as f:
    f.write(app_py)

print("Created deployment/app.py and deployment/requirements.txt")
print()
print("MANUAL DEPLOYMENT STEPS (Hugging Face Spaces — free, beginner-friendly):")
print("1. Go to https://huggingface.co/new-space")
print("2. Name it e.g. 'agentic-schedule-assistant', SDK = Gradio, hardware = free CPU basic.")
print("3. Upload deployment/app.py and deployment/requirements.txt to the Space repo")
print("   (via the web UI, or `git clone` the Space repo and `git push`).")
print("4. In the Space's Settings -> 'Repository secrets', add a secret named GOOGLE_API_KEY")
print("   with your Gemini API key as the value.")
print("5. Wait for the Space to build. Your persistent app URL will look like:")
print("   https://<your-username>-agentic-schedule-assistant.hf.space")


In [ ]:
# CELL 29 — Create deployed_url.txt with the REAL deployed URL
#
# IMPORTANT: we do NOT invent a URL. Complete the manual deployment steps from CELL 28
# first, then paste the actual live URL below before running this cell.

DEPLOYED_URL = ""  # <-- e.g. "https://your-username-agentic-schedule-assistant.hf.space"

if not DEPLOYED_URL:
    print(
        "⚠️  DEPLOYED_URL is empty. Deploy the app first (see CELL 28's instructions), "
        "then set DEPLOYED_URL above to the real, live URL and re-run this cell."
    )
else:
    with open("deployed_url.txt", "w") as f:
        f.write(DEPLOYED_URL)
    print(f"✅ Saved deployed_url.txt with: {DEPLOYED_URL}")
